In [510]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split


Load Data

In [511]:
data = datasets.load_iris()

In [512]:
X = data["data"]
y = data["target"]

In [513]:
y = np.eye(3, dtype=float)[y] # one hot encoding for the class

In [514]:
X.shape, y.shape

((150, 4), (150, 3))

Train-test split

In [515]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # makes the split reproducible
)


In [516]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((120, 4), (30, 4), (120, 3), (30, 3))

Create functions required by the neural network

In [517]:
# Parameters of the neural network, these are configurable
# for now we will hard code a few things, and then find a way to make this more configurable easily

# Only this needs to be changed if we are changing the architecture, assuming a 3 layer network
NEURONS_INFO = {0: 8, 1: 4, 2: 3} # E.g., {0: 3, 1: 2} -> first layer has 3 neurons, second layer has 2 neurons
N_LAYERS = len(NEURONS_INFO.keys())


In [518]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [519]:
def forward_pass(row, weights_matrices, bias_matrices):
    """Given a single row of training data,
    this function calculates the output produced by the network.
    
    The dimension of `row` is (<no. of features>, ).
    """
    row = row.reshape((1,-1))
    z_values = [0]*N_LAYERS
    a_values = [0]*N_LAYERS
    
    for layer in range(N_LAYERS):
        if layer == 0:
            # print(f"Layer {layer}: σ(X {row.shape} x W {weights_matrices[layer].shape} + B {bias_matrices[layer].T.shape}")
            z_values[layer] = row @ weights_matrices[layer] + bias_matrices[layer].T
        else:
            z_values[layer] = a_values[layer-1] @ weights_matrices[layer] + bias_matrices[layer].T
            # print(f"Layer {layer}: σ(A {a_values[layer-1].shape} x W {weights_matrices[layer].shape} + B {bias_matrices[layer].T.shape}")
        a_values[layer] = sigmoid(z_values[layer])
        

    return z_values, a_values

In [520]:
def train(X_train, y_train, epochs=2000, eta=0.05):
    training_size = X_train.shape[0]
    layer_sizes = [X_train.shape[1]] + [NEURONS_INFO[layer] for layer in range(N_LAYERS)]

    np.random.seed(42)
    weights_matrices = [
        np.random.uniform(-1, 1, (layer_sizes[layer], layer_sizes[layer + 1]))
        for layer in range(N_LAYERS)
    ]
    bias_matrices = [
        np.random.uniform(-1, 1, (layer_sizes[layer + 1], 1))
        for layer in range(N_LAYERS)
    ]

    for epoch in range(epochs):
        loss = 0.0
        dC_dw = [np.zeros_like(weights) for weights in weights_matrices]
        dC_db = [np.zeros_like(bias) for bias in bias_matrices]

        for X_i, y_i in zip(X_train, y_train):
            _, a_values = forward_pass(X_i, weights_matrices, bias_matrices)
            target = y_i.reshape(1, -1)
            output = a_values[-1]
            loss += np.sum((target - output) ** 2)

            deltas = [None] * N_LAYERS
            deltas[-1] = 2 * (output - target) * output * (1 - output)
            for layer in range(N_LAYERS - 2, -1, -1):
                activation = a_values[layer]
                deltas[layer] = (
                    deltas[layer + 1] @ weights_matrices[layer + 1].T
                ) * activation * (1 - activation)

            for layer in range(N_LAYERS):
                previous_activation = (
                    X_i.reshape(1, -1) if layer == 0 else a_values[layer - 1]
                )
                dC_dw[layer] += previous_activation.T @ deltas[layer]
                dC_db[layer] += deltas[layer].T

        loss /= training_size
        for layer in range(N_LAYERS):
            weights_matrices[layer] -= eta * dC_dw[layer] / training_size
            bias_matrices[layer] -= eta * dC_db[layer] / training_size

        if (epoch + 1) % 100 == 0:
            print(f"===============Epoch {epoch + 1}/{epochs}===============")
            print(f"Loss: {loss}\n")

    return weights_matrices, bias_matrices

Training the network

In [521]:
trained_weights_matrices, trained_bias_matrices = train(X_train, y_train)

===============Epoch 100/2000===============
Loss: 0.7070449742325564

===============Epoch 200/2000===============
Loss: 0.6639060775732174

===============Epoch 300/2000===============
Loss: 0.6571324149947437

===============Epoch 400/2000===============
Loss: 0.6526094299748617

===============Epoch 500/2000===============
Loss: 0.6459365019770781

===============Epoch 600/2000===============
Loss: 0.635753128850872

===============Epoch 700/2000===============
Loss: 0.6233947475896157

===============Epoch 800/2000===============
Loss: 0.6081897270629616

===============Epoch 900/2000===============
Loss: 0.5890339707368457

===============Epoch 1000/2000===============
Loss: 0.5657177941677033

===============Epoch 1100/2000===============
Loss: 0.5390997301263101

===============Epoch 1200/2000===============
Loss: 0.510972673238136

===============Epoch 1300/2000===============
Loss: 0.48348638059311344

===============Epoch 1400/2000===============
Loss: 0.45841013349916443

=

Predictions on test data 

In [522]:
y_pred = np.zeros(y_test.shape)
test_size = y_pred.shape[0]

In [523]:
y_pred = np.zeros(y_test.shape, dtype=int)
test_size = y_pred.shape[0]

for i in range(test_size):
    _, activated_outputs = forward_pass(X_test[i], trained_weights_matrices, trained_bias_matrices)
    pred = activated_outputs[-1].flatten()
    one_hot = np.zeros_like(y_pred[i])
    one_hot[np.argmax(pred)] = 1
    y_pred[i] = one_hot

In [524]:
y_pred.shape, y_test.shape

((30, 3), (30, 3))

Accuracy

In [525]:
np.sum(np.all(y_pred == y_test, axis=1))/test_size

np.float64(0.8)